# Route Resilience — Road Segmentation Training (Kaggle)

## ⚠️ Required Kaggle Settings (set BEFORE running any cell)

1. **Settings → Accelerator → GPU T4 x2** (recommended) or **GPU P100** (simpler single-GPU).
   - **GPU T4 x2**: Two T4s (2×16 GB VRAM). PyTorch will use the first GPU by default; enable `nn.DataParallel` if you want to split batches across both.
   - **GPU P100**: Single GPU (16 GB VRAM). Simpler — no multi-GPU code needed. Plenty fast for a ResNet34-backbone U-Net at 512×512 tiles.
   - **Without this setting, training silently falls back to CPU (~50× slower).**

2. **Settings → Internet → On** (REQUIRED). Internet access must be enabled to download packages like `segmentation-models-pytorch` that are not pre-installed in the base environment.

## Attached Datasets
- **DeepGlobe Road Extraction** — RGB satellite tiles + RGB class masks + `class_dict.csv`
- **HybridSAR Road Dataset (HSRD)** — SAR imagery (deferred to a future training run)

## Step 0 — Discover actual dataset folder names
Run this cell **first**. Copy the exact folder name(s) printed below into the `DEEPGLOBE_DIR` / `HSRD_DIR` constants in the next cell.

In [ ]:
import os
print('Datasets mounted under /kaggle/input:')
for entry in sorted(os.listdir('/kaggle/input')):
    full = os.path.join('/kaggle/input', entry)
    n_items = len(os.listdir(full)) if os.path.isdir(full) else 0
    print(f'  {entry}/   ({n_items} items)')

In [ ]:
# ============================================================
# CONFIG — Exact Hardcoded Paths
# ============================================================
import os

# ---- Dataset paths (exact confirmed slugs) ----
SRC_DATASET_PATH = '/kaggle/input/datasets/satyveeryadav/route-resillence-src/route-resillence-src'
DEEPGLOBE_DIR = '/kaggle/input/deepglobe-road-extraction-dataset'
HSRD_DIR = '/kaggle/input/the-hybridsar-road-dataset-hsrd'

# ---- Training hyperparameters ----
TILE_SIZE = 512
BATCH_SIZE = 8          # Reduce to 4 if OOM on T4 (16 GB VRAM)
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4

# Model: DeepGlobe = RGB = in_channels=3, no change needed
ARCHITECTURE = 'Unet'   # Options: 'Unet', 'UnetPlusPlus', 'DeepLabV3Plus'
ENCODER = 'resnet34'
IN_CHANNELS = 3         # DeepGlobe is standard RGB

# ---- Output paths ----
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# CONDITIONAL INSTALLS
# Kaggle pre-installs PyTorch + CUDA — do NOT reinstall torch.
# Only install packages missing from the Kaggle base image.
# ============================================================

def _ensure(pkg_import, pip_name=None):
    try:
        __import__(pkg_import)
    except ImportError:
        import subprocess, sys
        pkg = pip_name or pkg_import
        print(f"Installing {pkg}...")
        # Base install command
        cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check']
        
        # Handle PEP 668 Externally Managed Environments on Python 3.12+
        if sys.version_info >= (3, 12):
            cmd.append('--break-system-packages')
            
        cmd.append(pkg)
        
        # Run and capture output to print exact error on failure
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print("\n" + "!"*60)
            print(f"CRITICAL ERROR: Failed to install '{pkg}'.")
            print("--- PIP STDOUT ---")
            print(result.stdout)
            print("--- PIP STDERR ---")
            print(result.stderr)
            print("\nPOSSIBLE SOLUTIONS:")
            print("1. Turn on Internet in your Kaggle Notebook Settings (right sidebar -> Settings -> Internet -> On).")
            print("2. Make sure you are using a GPU accelerator (T4 or P100).")
            print("!"*60 + "\n")
            raise RuntimeError(f"Failed to install {pkg}: {result.stderr}")
        else:
            print(f"Successfully installed {pkg}.")

_ensure('segmentation_models_pytorch', 'segmentation-models-pytorch')
_ensure('albumentations')
print('All packages ready.')

In [ ]:
# ============================================================
# IMPORT PROJECT MODULES
# ============================================================
import os
import sys

# Auto-detect exact source directory (handles nested, flat, and user-specific layouts)
possible_paths = [
    '/kaggle/input/datasets/satyveeryadav/route-resillence-src/route-resillence-src',
    '/kaggle/input/route-resillence-src/route-resillence-src',
    '/kaggle/input/route-resillence-src'
]
SRC_DATASET_PATH = None
for path in possible_paths:
    if os.path.exists(os.path.join(path, 'data_pipeline.py')):
        SRC_DATASET_PATH = path
        break

if SRC_DATASET_PATH is not None:
    print(f"✅ Source path detected: {SRC_DATASET_PATH}")
    if SRC_DATASET_PATH not in sys.path:
        sys.path.insert(0, SRC_DATASET_PATH)
else:
    print("❌ ERROR: data_pipeline.py not found in any of the possible paths.")
    if os.path.exists('/kaggle/input/route-resillence-src'):
        print("Files in '/kaggle/input/route-resillence-src':", os.listdir('/kaggle/input/route-resillence-src'))
    else:
        print("Dataset directory '/kaggle/input/route-resillence-src' is missing.")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
from torch.utils.data import DataLoader

from data_pipeline import (
    fetch_osm_roads,
    rasterize_osm_to_mask,
    tile_image_and_mask,
    add_synthetic_occlusion,
    build_binary_road_mask,
)
from segmentation_train import (
    RoadDataset,
    build_model,
    train_model,
    predict_mask,
    compute_metrics,
    get_train_augmentations,
    get_val_augmentations,
)
from geo_utils import assert_projected_crs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU not detected! Enable in Settings → Accelerator.')

In [ ]:
# ============================================================
# PATH SANITY CHECK
# ============================================================
import os
assert os.path.exists(SRC_DATASET_PATH), f"src path not found: {SRC_DATASET_PATH}"
assert os.path.exists(os.path.join(SRC_DATASET_PATH, 'segmentation_train.py')), "segmentation_train.py missing from src dataset"
assert os.path.exists(DEEPGLOBE_DIR), f"DeepGlobe path not found: {DEEPGLOBE_DIR}"
print("✅ All paths verified, ready to proceed.")

In [ ]:
# ============================================================
# EXPLORE DEEPGLOBE DATASET STRUCTURE
# ============================================================

print('DeepGlobe directory contents:')
for entry in sorted(os.listdir(DEEPGLOBE_DIR)):
    full = os.path.join(DEEPGLOBE_DIR, entry)
    if os.path.isdir(full):
        print(f'  {entry}/  ({len(os.listdir(full))} files)')
    else:
        print(f'  {entry}  ({os.path.getsize(full)} bytes)')

# Check class_dict.csv
class_dict_path = os.path.join(DEEPGLOBE_DIR, 'class_dict.csv')
if os.path.exists(class_dict_path):
    class_dict_df = pd.read_csv(class_dict_path)
    print('\nclass_dict.csv:')
    print(class_dict_df.to_string())
else:
    print('\n⚠️  class_dict.csv not found — check DEEPGLOBE_DIR path')

# Peek at a sample image to check tile dimensions
train_dir = os.path.join(DEEPGLOBE_DIR, 'train')
sample_sats = sorted(glob(os.path.join(train_dir, '*_sat.*')))
if sample_sats:
    sample_img = Image.open(sample_sats[0])
    print(f'\nSample tile: {os.path.basename(sample_sats[0])}')
    print(f'  Size: {sample_img.size} (W x H), Mode: {sample_img.mode}')
    print(f'  Total train sat images: {len(sample_sats)}')
else:
    print('\n⚠️  No *_sat.* files found in train/ — check dataset structure')

# Check test/ directory contents and mask availability
test_dir = os.path.join(DEEPGLOBE_DIR, 'test')
if os.path.isdir(test_dir):
    test_files = os.listdir(test_dir)
    test_masks = [f for f in test_files if '_mask' in f]
    print(f'\nTest directory: {test_dir}')
    print(f'  Total files: {len(test_files)}')
    print(f'  Mask files found: {len(test_masks)}')
    # Note: Since no masks exist in test/, it is unusable for supervised training/validation and should be ignored.
else:
    print('\nTest directory not found or is not a directory.')

In [ ]:
# ============================================================
# LOAD DEEPGLOBE DATA
# ============================================================
# DeepGlobe convention: {id}_sat.jpg = image, {id}_mask.png = RGB class mask
# We use class_dict.csv to extract road class → binary mask.

def load_deepglobe_split(split_dir, class_dict_df, tile_size=512):
    """
    Load all image/mask pairs from a DeepGlobe split directory.
    Returns lists of (HxWxC image, HxW binary mask) numpy arrays.
    If tiles are larger than tile_size, they are center-cropped;
    if smaller or equal, they are used as-is.
    """
    # Find satellite images (try common extensions)
    sat_files = sorted(glob(os.path.join(split_dir, '*_sat.jpg')))
    if not sat_files:
        sat_files = sorted(glob(os.path.join(split_dir, '*_sat.png')))
    if not sat_files:
        sat_files = sorted(glob(os.path.join(split_dir, '*_sat.*')))

    images, masks = [], []
    skipped = 0

    for sat_path in sat_files:
        # Derive mask path: replace _sat with _mask, try .png then .jpg
        base = sat_path.replace('_sat.', '_mask.')
        mask_path = base.replace('.jpg', '.png')  # masks are usually PNG
        if not os.path.exists(mask_path):
            mask_path = base  # try same extension
        if not os.path.exists(mask_path):
            skipped += 1
            continue

        img = np.array(Image.open(sat_path).convert('RGB'))
        mask_rgb = np.array(Image.open(mask_path).convert('RGB'))

        # Convert RGB class mask to binary road mask
        binary_mask = build_binary_road_mask(mask_rgb, class_dict_df)

        h, w = img.shape[:2]

        if h > tile_size or w > tile_size:
            # Center-crop to tile_size x tile_size
            r0 = (h - tile_size) // 2
            c0 = (w - tile_size) // 2
            img = img[r0:r0+tile_size, c0:c0+tile_size]
            binary_mask = binary_mask[r0:r0+tile_size, c0:c0+tile_size]
        elif h < tile_size or w < tile_size:
            # Pad to tile_size (rare for DeepGlobe, but handle gracefully)
            pad_img = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
            pad_msk = np.zeros((tile_size, tile_size), dtype=np.uint8)
            pad_img[:h, :w] = img
            pad_msk[:h, :w] = binary_mask
            img, binary_mask = pad_img, pad_msk

        images.append(img)
        masks.append(binary_mask)

    print(f'  Loaded {len(images)} pairs from {split_dir}' +
          (f' (skipped {skipped} — no matching mask)' if skipped else ''))
    return images, masks

# ---- Load train and val splits ----
print('Loading training split...')
train_imgs, train_msks = load_deepglobe_split(
    os.path.join(DEEPGLOBE_DIR, 'train'), class_dict_df, tile_size=TILE_SIZE)

# Use DeepGlobe's provided valid/ split directly (no re-splitting needed)
valid_dir = os.path.join(DEEPGLOBE_DIR, 'valid')
if os.path.isdir(valid_dir) and len(os.listdir(valid_dir)) > 0:
    print('Loading validation split...')
    val_imgs, val_msks = load_deepglobe_split(
        valid_dir, class_dict_df, tile_size=TILE_SIZE)
else:
    # Fallback: split train 85/15 if no valid/ directory
    print('No valid/ directory found — splitting train 85/15...')
    n = len(train_imgs)
    n_val = max(1, int(n * 0.15))
    rng = np.random.default_rng(42)
    indices = rng.permutation(n)
    val_imgs   = [train_imgs[i] for i in indices[:n_val]]
    val_msks   = [train_msks[i] for i in indices[:n_val]]
    train_imgs = [train_imgs[i] for i in indices[n_val:]]
    train_msks = [train_msks[i] for i in indices[n_val:]]

print(f'\nFinal: {len(train_imgs)} train / {len(val_imgs)} val tiles')

# Quick sanity: show road pixel coverage
road_pcts = [m.mean()*100 for m in train_msks[:20]]
print(f'Road pixel coverage (first 20 train tiles): '
      f'mean={np.mean(road_pcts):.1f}%, range=[{min(road_pcts):.1f}, {max(road_pcts):.1f}]%')

In [ ]:
# ============================================================
# DATALOADERS
# ============================================================

train_ds = RoadDataset(
    train_imgs, train_msks,
    augmentations=get_train_augmentations(TILE_SIZE),
    apply_occlusion=True,   # synthetic occlusion for robustness
)
val_ds = RoadDataset(
    val_imgs, val_msks,
    augmentations=get_val_augmentations(),
    apply_occlusion=False,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} tiles ({len(train_loader)} batches) | '
      f'Val: {len(val_ds)} tiles ({len(val_loader)} batches)')

In [ ]:
# ============================================================
# BUILD MODEL + TRAIN
# ============================================================

model = build_model(
    architecture=ARCHITECTURE,
    encoder_name=ENCODER,
    encoder_weights='imagenet',
    in_channels=IN_CHANNELS,  # DeepGlobe = RGB = 3, confirmed
    classes=1,
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
)

In [ ]:
# ============================================================
# VISUAL SANITY CHECK — Inference on validation tiles
# ============================================================

best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.to(device)
model.eval()

NUM_EXAMPLES = min(5, len(val_imgs))
fig, axes = plt.subplots(NUM_EXAMPLES, 3, figsize=(14, 4 * NUM_EXAMPLES))
if NUM_EXAMPLES == 1:
    axes = axes[np.newaxis, :]

for i in range(NUM_EXAMPLES):
    img = val_imgs[i]
    gt  = val_msks[i]
    pred = predict_mask(model, img, device=device, threshold=0.5)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Satellite Image')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(pred, cmap='gray')
    axes[i, 1].set_title('Predicted Mask')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(gt, cmap='gray')
    axes[i, 2].set_title('Ground Truth')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/val_predictions.png', dpi=150)
plt.show()
print('Saved to /kaggle/working/val_predictions.png')

In [ ]:
# ============================================================
# FINAL METRICS SUMMARY
# ============================================================
import json

print('='*60)
print('TRAINING SUMMARY')
print('='*60)
print(f'Dataset      : DeepGlobe Road Extraction')
print(f'Architecture : {ARCHITECTURE} / {ENCODER}')
print(f'Tile size    : {TILE_SIZE}x{TILE_SIZE}')
print(f'Epochs       : {NUM_EPOCHS}')
print(f'Best Val IoU : {max(history["val_iou"]):.4f}  (epoch {np.argmax(history["val_iou"])+1})')
print(f'Best Val Dice: {max(history["val_dice"]):.4f}')
print(f'Final Train Loss : {history["train_loss"][-1]:.4f}')
print(f'Final Val Loss   : {history["val_loss"][-1]:.4f}')
print('='*60)

# Save metrics for dashboard consumption
metrics = {
    'iou': float(max(history['val_iou'])),
    'dice': float(max(history['val_dice'])),
    'best_epoch': int(np.argmax(history['val_iou']) + 1),
    'dataset': 'DeepGlobe',
    'architecture': f'{ARCHITECTURE}/{ENCODER}',
}
metrics_path = '/kaggle/working/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nMetrics saved to {metrics_path}')

# Confirm checkpoints exist
for fname in ['best_model.pth', 'final_model.pth']:
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'✓ {fname} saved ({size_mb:.1f} MB) → {fpath}')
    else:
        print(f'✗ {fname} NOT FOUND at {fpath}')

print()
print('⚠️  REMINDER: Download checkpoints + metrics.json from Output panel')
print('   before closing this session. /kaggle/working/ does NOT persist')
print('   unless you Save Version or download manually.')

## Next Steps

1. **Download** `best_model.pth` and `metrics.json` from the **Output** panel (right sidebar → `/kaggle/working/`).
2. Place `metrics.json` in `outputs/` for the Streamlit dashboard to consume.
3. Use `best_model.pth` to run inference → `skeleton_to_graph.py` → `graph_healing.py` → `centrality.py` → dashboard.
4. If IoU < 0.50: try `UnetPlusPlus` or `DeepLabV3Plus`, increase `NUM_EPOCHS` to 50, or reduce `TILE_SIZE` to 256.
5. **SAR training** (HybridSAR dataset): requires changing `IN_CHANNELS` and normalization — see TODO in config cell.